# Approx Lab

The goal of this lab is to gain experience using approximation algorithms.

For the purpose of this assignment, you will be implementing the knapsack problem. Before continuing, the following cell contains any dependencies you may need for this assignment.

In [1]:
import pandas as pd

## Background Info

### Knapsack Problem
We are actually going to be looking at a specific (hopefully fun) implementation of the knapsack problem.

The traditional knapsack problem asks you to pack a set of items, with given weights and values into a "knapsack" with a maximum capacity. You cannot pack more items than the capacity of the knapsack. The goal is to select a subset of items that maximize the total value with what can fit in the knapsack. The name "knapsack" comes from the similarity to a hiker trying to pack a bag without going over a certain weight.

The knapsack problem is an NP-Hard optimization problem, which is why it is one of the classic approximation algorithms.

### Specifications for this Lab
The variation that we will be doing in this assignment is referred to as an 0/1 knapsack problem, meaning that items must be completely put in the knapsack or they must be left behind (you can't have a fractional portion of an item). We will be representing this as a simple fantasy sports league - specifically Formula 1 drivers. 

You are provided with a list of F1 drivers and constructors with given costs and points as well as a budget of $100M. You will be selecting five drivers and two constructors (teams). 

### Scoring Information
Points are gained from qualification results for drivers and constructors as well as race results for drivers and constructors. 

<!-- Qualifications
- Drivers score based on their resulting position 
- Constructor score the combined total of their two drivers in qualifying plus a modifier based on how far both their drivers get. -->
<!-- 
| Drivers    | Points |
| -------- | ------- |
| Pole Position (1st Place) | 10 |
| 2nd Place | 9 |
| 3rd Place | 8 |
| 4th Place | 7 |
| 5th Place | 6 |
| 6th Place | 5 |
| 7th Place | 4 |
| 8th Place | 3 |
| 9th Place | 2 |
| 10th Place | 1 |
| 11th-20th Place | 0 |
| No Time Set | 0 |
| Disqualification | 0 | -->


<!-- | Constructors | Points |
| ----------- | ------- |
| Neither Driver Reaches Q2 | -1 |
| One Driver Reaches Q2 | 1 |
| Both Drivers Reach Q2 | 5 |
| Both Drivers Reach Q3 | 10 | -->


<!-- Race
- Drivers score based on their resulting position, position difference from starting order, and some additional bonuses
- Constructors score a combined total of their two drivers' results (excluding the Driver of the Day bonus) ~~plus a modifier based on pitstop times~~ -->

<!-- | Drivers | Points |
| -------- | ------- |
| Positions Gained | 1/position |
| Positions Lost | -1/position |
| Overtakes Made | 1/overtake |
| Fastest Lap | 10 |
| Driver of the Day | 10 | -->

<!-- | Race Result | Points |
| -------- | ------- |
| 1st Place | 25 |
| 2nd Place | 18 |
| 3rd Place | 15 |
| 4th Place | 12 |
| 5th Place | 10 |
| 6th Place | 8 |
| 7th Place | 6 |
| 8th Place | 4 |
| 9th Place | 2 |
| 10th Place | 1 |
| 11th-20th Place | 0 |
| DNF | -20 |
| Disqualification | -20 | -->

Fortunately for you, we have access to the current point standings post-2025 China GP for drivers & constructors, which are available in `drivers.csv` & `constructors.csv`.

*Note: The costs are in millions, so 30 is actually 30 M.

### Questions (Answered in Various Parts of the Lab): 
1. What is the general setup of the problem?
2. Determining the mathematical functions of the problem
3. Implementing a greedy algorithm


## A. Assessing the Problem

The values that are provided for this lab are from the points distribution PRIOR to the Japan Grand Prix (which means we can actually compare with the race results at the submission deadline!)

Create a fantasy team https://fantasy.formula1.com/en/create-team for the upcoming Grand Prix to get a better idea of what drivers, constructors, points, and the budget are. Who is on your team & how much did you spend? Don't spend too much time on this, it really is just to get familiar with the format.

*I picked Norris, Ocon, Hulkenberg, Bearman, and Lawson for my drivers and McLaren and Haas for my constructors. In total I spent $99.2M for 409 points.*

What represents the "items" in the knapsack? 

*The "items" are the drivers and constructors.*

What represents the weights?

*Their weights are their cost to purchase.*

What represents the values?

*Their values are how many points they earn.*

In [78]:
# Load the data from the CSV files into a format that you are comfortable working with
# This can be arrays, dictionaries, or pd DataFrames

drivers = pd.read_csv('drivers.csv')
constructors = pd.read_csv('constructors.csv')

# Print/Display your data as well to confirm that you have successfully saved it.
print(drivers)
print(constructors)

                   Driver  Points  Cost
0            Lando Norris     100  29.6
1   Andrea Kimi Antonelli      61  19.3
2          George Russell      60  21.4
3          Max Verstappen      59  28.6
4           Oscar Piastri      55  23.0
5            Lance Stroll      33   9.3
6            Esteban Ocon      32   8.1
7          Alexaner Albon      28  12.8
8         Nico Hulkenberg      22   7.6
9          Oliver Bearman      22   6.7
10         Lewis Hamilton       3  23.6
11           Yuki Tsunoda       0  16.8
12            Liam Lawson       0   8.4
13        Charles LeClerc       0  25.3
14           Pierre Gasly      -7  10.6
15           Isack Hadjar      -9   5.0
16            Jack Doohan      -9   6.0
17           Carlos Sainz     -11  11.9
18      Gabriel Bortoleto     -11   4.8
19        Fernando Alonso     -36   7.6
       Constructor  Points  Cost
0          McLaren     172  30.6
1         Mercedes     136  23.3
2  Red Bull Racing      86  25.4
3             Haas      61  

# B. Integer Program

Before implementing our approx algorithm, we need to first consider the integer program. 

Given the following variables, create an integer program that maximizes the total points of drivers while remaining under the budget: 
- $x_i$ is the decision variable that determines whether driver $i$ is included in the team or not
- $p_i$ is the # of points that driver $i$ is worth
- $c_i$ is the cost of driver $i$

For now, we will focus on just the drivers and ignore the constructors. We will also temporarily reduce our budget to **$50M**

While we are not planning on implementing the full integer program, it helps us quantify parts of the problem


$\text{maximize} \quad \sum_{i=1}^{n} p_i x_i$  \
$\text{subject to} \quad \sum_{i=1}^{n} c_i x_i \le 50 \text{ and } x_i \in \{0, 1\}$ 


## Decision Variable & Constraints

We can make helper functions for our future algorithm based on constraints that we derive when forming the IP: 
1. Which drivers should be included in our solution? 
2. What is our maximum cost?
3. What is our maximum # of drivers? 

In [75]:
budget = 50 # in millions

def within_maximum_cost(cost) -> bool:
    return cost <= budget

def at_maximum_drivers(drivers: list) -> bool:
    return len(drivers) >= 5

# C. Greedy Algorithm


There are multiple ways we can go about solving the knapsack problem. For this lab, we will be focusing on a greedy algorithm and a variant of that.

Let's start with a basic greedy algorithm:
1. Calculate the value-to-cost ratio for each driver 
2. Sort drivers in non-increasing order of value-to-cost
2. Greedily select drivers from list until either budget or max drivers constraint is hit

Use the helper methods from the previous part in your implementation.

In [76]:
def value_to_cost(df: pd.DataFrame) -> pd.DataFrame:
    new_df = df.copy()
    new_df['ValueToCost'] = df['Points'] / df['Cost']
    return new_df

def sort_items(df: pd.DataFrame) -> pd.DataFrame: 
    return df.sort_values(by='ValueToCost', ascending=False)

def greedy_driver_select(drivers: pd.DataFrame) -> list:

    with_value_to_cost = value_to_cost(drivers)
    sorted = sort_items(with_value_to_cost)
    
    total_cost: float = 0
    selected: list = []
    for driver in sorted.itertuples():

        if within_maximum_cost(total_cost + driver.Cost): # type: ignore
            selected.append(driver)
            total_cost += driver.Cost # type: ignore

        if at_maximum_drivers(selected):
            break
    
    return selected

for driver in greedy_driver_select(drivers):
    print(driver)


Pandas(Index=6, Driver='Esteban Ocon', Points=32, Cost=8.1, ValueToCost=3.9506172839506175)
Pandas(Index=5, Driver='Lance Stroll', Points=33, Cost=9.3, ValueToCost=3.548387096774193)
Pandas(Index=0, Driver='Lando Norris', Points=100, Cost=29.6, ValueToCost=3.378378378378378)


Try running your basic greedy algorithm. 

You may notice a few flaws:
1. The greedy algorithm is only looking at value-to-cost ratio, which may miss out on an overall improvement in value that still fits within the budget
2. Depending on your implementation, you may not currently be meeting the "5 total drivers" constraint - this is fine


## Redux Greedy Algorithm
We are going to next implement a small adjustment to this algorithm in order to improve its performance. \
This is a 2-approximation for the knapsack problem. 

1. Calculate the value-to-cost ratio for each driver
2. Sort drivers in non-increasing order of value-to-cost
3. Greedily add items until we hit an item that is too big
4. Compare the too big item against the already existing list of items & pick the better of the two

It might help to make a separate function to handle #4. 

In [ ]:
def redux_greedy_driver_select(drivers) -> list:

    with_value_to_cost = value_to_cost(drivers)
    sorted = sort_items(with_value_to_cost)
    
    total_cost: float = 0
    total_value: float = 0
    selected: list = []
    for driver in sorted.itertuples():

        if within_maximum_cost(total_cost + driver.Cost) and not at_maximum_drivers(selected): # type: ignore
            selected.append(driver)
            total_cost += driver.Cost # type: ignore
            total_value += driver.Points # type: ignore
        else:
            for i in range(len(selected)):
                # Replace item with new driver
                possible_cost = total_cost - selected[i].Cost + driver.Cost
                possible_value = total_value - selected[i].Points + driver.Points

                # If it's better, switch to that
                if within_maximum_cost(possible_cost) and possible_value > total_value:
                    selected.pop(i)
                    selected.append(driver)
                    total_cost = possible_cost
                    total_value = possible_value

    return selected

for driver in redux_greedy_driver_select(drivers):
    print(driver)

Pandas(Index=6, Driver='Esteban Ocon', Points=32, Cost=8.1, ValueToCost=3.9506172839506175)
Pandas(Index=5, Driver='Lance Stroll', Points=33, Cost=9.3, ValueToCost=3.548387096774193)
Pandas(Index=0, Driver='Lando Norris', Points=100, Cost=29.6, ValueToCost=3.378378378378378)


# D. Reintroducing Constructors

We've been neglecting our constructors so far and cutting our budget. Let's make them feel included in our processes again. 

## Integer Program

We should start by redefining our original IP to see what constraints we need to add. \
We've added some more variables to our IP: 
- $x_j$ is the decision variable that determines whether constructor $j$ is included in the team or not 
- $p_j$ is the # of points that constructor $j$ is worth
- $c_j$ is the cost of constructor $j$

Our budget has also been increased back to **100M**. 


$\text{maximize} \quad \sum_{i=1}^{D} p_i x_i + \sum_{j=1}^{C} p_j x_j$  \
$\text{subject to} \quad \sum_{i=1}^{D} c_i x_i + \sum_{j=1}^{C} c_j x_j \le 100 \text{ and } x_i, x_j \in \{0, 1\}$ 

## Constraints 

To redefine our greedy algorithm to include constructors, we have to also write helpers that correspond to our new constraints.

In [86]:
budget = 100 # in millions
maximum_drivers_f1 = 5
maximum_constructors_f1 = 2

def is_driver(item) -> bool:
    return not pd.isna(item.Driver)

def at_maximum_drivers_f1(selected: list) -> bool:
    return sum([1 if is_driver(item) else 0 for item in selected]) >= maximum_drivers_f1

def at_maximum_constructors_f1(selected: list) -> bool:
    return sum([1 if not is_driver(item) else 0 for item in selected]) >= maximum_constructors_f1

def above_maximum_drivers_f1(selected: list) -> bool:
    return sum([1 if is_driver(item) else 0 for item in selected]) > maximum_drivers_f1

def above_maximum_constructors_f1(selected: list) -> bool:
    return sum([1 if not is_driver(item) else 0 for item in selected]) > maximum_constructors_f1

## Greedy Algorithm

Update the greedy algorithm to calculate for buying both constructors and drivers. \
Remember, your list of non-increasing value-to-cost ratios should be a combination of both drivers & constructors. 

In [ ]:
def greedy_f1(drivers, constructors) -> list:

    # Combine the dataframes
    combined = pd.merge(drivers, constructors, how='outer')

    # Calculate value to cost ratio and sort the dataframe
    with_value_to_cost = value_to_cost(combined)
    sorted = sort_items(with_value_to_cost)
    
    total_cost: float = 0
    total_value: float = 0
    selected: list = []
    for item in sorted.itertuples():

        # Check if we are at the limit of drivers or constructors based on the type of chosen item
        at_maximum = at_maximum_drivers_f1(selected) if is_driver(item) else at_maximum_constructors_f1(selected)

        # If we can add another item without removing an item, add it
        if within_maximum_cost(total_cost + item.Cost) and not at_maximum: # type: ignore
            selected.append(item)
            total_cost += item.Cost # type: ignore
            total_value += item.Points # type: ignore
        else:
            for i in range(len(selected)):

                # Replace item with new item
                possible_cost = total_cost - selected[i].Cost + item.Cost
                possible_value = total_value - selected[i].Points + item.Points

                possible = selected.copy()
                possible.pop(i)
                possible.append(item)

                # If it's better, switch to that
                valid: bool = within_maximum_cost(possible_cost) and not above_maximum_drivers_f1(possible) and not above_maximum_constructors_f1(possible)
                better = possible_value > total_value
                if valid and better:
                    selected = possible
                    total_cost = possible_cost
                    total_value = possible_value

    return selected

for item in greedy_f1(drivers, constructors):
    print(item)



Pandas(Index=28, Driver=nan, Points=136, Cost=23.3, Constructor='Mercedes', ValueToCost=5.836909871244635)
Pandas(Index=29, Driver=nan, Points=172, Cost=30.6, Constructor='McLaren', ValueToCost=5.620915032679738)
Pandas(Index=19, Driver='Lance Stroll', Points=33, Cost=9.3, Constructor=nan, ValueToCost=3.548387096774193)
Pandas(Index=27, Driver='Lando Norris', Points=100, Cost=29.6, Constructor=nan, ValueToCost=3.378378378378378)
Pandas(Index=14, Driver='Oliver Bearman', Points=22, Cost=6.7, Constructor=nan, ValueToCost=3.2835820895522385)
